# File POTs and LaTeX sample tables

Collects the POT, number of events (`wcpselection/T_eval` entries, as loaded by `create_df.py`), and trigger counts (for data/EXT) of every ROOT file listed in `download_input_files.sh`, and uses them to generate the sample tables in `temp_latex_file_tables.tex`.

* Simulated samples (overlays, NuWro, DetVar): sum of `wcpselection/T_pot/pot_tor875good`, the same as `_get_file_metadata` in `src/create_df.py`.
* Data: `tor875_wcut` POT and `E1DCNT_wcut` triggers from `src/pot_and_trigger_numbers.py` (made with Zarko's `getDataInfo.py`, see `pot_and_trigger_processing.ipynb`).
* EXT: `EXT` triggers from `src/pot_and_trigger_numbers.py`. The POT is a POT-equivalent: EXT triggers times the POT per trigger of the paired data period (`ext_pot_normalizing_period`), as in `create_df.py`.

The last cell rewrites only the part of the `.tex` file between the `% BEGIN AUTO-GENERATED FILE TABLES` and `% END AUTO-GENERATED FILE TABLES` markers.

In [ ]:
# Tested using this python environment:
# source /nevis/houston/home/leehagaman/uv_base/bin/activate

import os
import re
import sys

import numpy as np
import pandas as pd
import uproot

sys.path.insert(0, "../src")
from file_locations import data_files_location
from pot_and_trigger_numbers import (
    open_data_POT, open_data_num_triggers, ext_num_triggers, ext_pot_normalizing_period,
)

download_script = "../download_input_files.sh"
tex_file = "../temp_latex_file_tables.tex"

In [ ]:
# every file path in the download script (commented-out lines are skipped)
remote_paths = re.findall(r'^\s*"(/[^"]+\.root)"', open(download_script).read(), re.M)
print(f"{len(remote_paths)} files in {download_script}")

DETVAR_VARIATIONS = {  # filename tag -> label
    "cv": "CV", "lya": "LYAttenuation", "lyd": "LYDown", "lyr": "LYRayleigh", "recomb2": "Recombination",
    "sce": "SCE", "WMX": "WireMod X", "WMYZ": "WireMod YZ",
    "WMthetaXZ": "WireMod thetaXZ", "WMthetaYZ": "WireMod thetaYZ",
}

def sample_type(base):
    for tag, name in [("beam_on", "data"), ("beam_off", "ext"), ("nuwro_overlay", "nuwro"),
                      ("intrinsic_nue_overlay", "nue_overlay"), ("_nu_overlay", "nu_overlay"),
                      ("NCpi0_overlay", "nc_pi0_overlay"), ("CCpi0_overlay", "numucc_pi0_overlay"),
                      ("dirt_overlay", "dirt_overlay"), ("delete_one_gamma", "del1g_overlay"),
                      ("isotropic_one_gamma", "iso1g_overlay")]:
        if tag in base:
            return name
    raise ValueError(f"unknown sample type: {base}")

def run_period(base):
    if "_3b_1mil_" in base or "_3b_500k_" in base:
        return "3b"
    if "_Run4b_" in base:
        return "4b"
    if "_cv_13a_" in base and base.endswith("_hist_3.root"):  # the 13a CV batch was split into run 1 and run 3a
        return "3a"
    if "_hist_opendata_19550.root" in base:  # run 4a open data
        return "4a"
    m = re.search(r"_hist_(\d[a-d]*)(_\de19opendata|_v\d+|_filtered)?\.root$", base)
    if m:
        return m.group(1)
    raise ValueError(f"unknown run period: {base}")

def detvar_variation(base):
    if "_3b_1mil_" in base: return "CV (3b 1M)"
    if "_3b_500k_" in base: return "CV (3b 500k)"
    if "_cv_13a_" in base: return "CV"
    return DETVAR_VARIATIONS[re.search(r"_overlay_([A-Za-z0-9]+)_surprise", base).group(1)]

def T_pot_sum_and_n_events(base):
    with uproot.open(f"{data_files_location}/{base}") as f:
        T_pot_sum = float(np.sum(f["wcpselection/T_pot"]["pot_tor875good"].array(library="np")))
        return T_pot_sum, f["wcpselection/T_eval"].num_entries

rows = []
for path in remote_paths:
    base = os.path.basename(path)
    sample = sample_type(base)
    run = run_period(base)
    is_detvar = "DetVar" in base
    T_pot_sum, n_events = T_pot_sum_and_n_events(base)
    row = {"sample": sample, "detvar": is_detvar, "variation": detvar_variation(base) if is_detvar else "",
           "run": run, "n_events": n_events, "num_triggers": np.nan, "T_pot_sum": T_pot_sum, "path": path}
    if sample == "data":
        row["POT"] = open_data_POT[run]
        row["num_triggers"] = open_data_num_triggers[run]
    elif sample == "ext":
        norm = ext_pot_normalizing_period[run]
        row["POT"] = ext_num_triggers[run] * open_data_POT[norm] / open_data_num_triggers[norm]
        row["num_triggers"] = ext_num_triggers[run]
    else:
        row["POT"] = row["T_pot_sum"]
    rows.append(row)

pot_df = pd.DataFrame(rows)
pot_df["file"] = pot_df["path"].map(os.path.basename)

In [ ]:
# readable summary
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 250):
    print(pot_df[["sample", "detvar", "variation", "run", "POT", "n_events", "num_triggers", "file"]].to_string(
        index=False, na_rep="",
        formatters={"POT": "{:.4e}".format,
                    "n_events": "{:,}".format, "num_triggers": lambda x: f"{int(x):,}"}))

In [ ]:
RUN_ORDER = ["1", "2", "3", "3a", "3b", "4a", "4b", "4bcd", "4c", "4d", "5"]
VARIATION_ORDER = ["CV", "CV (3b 1M)", "CV (3b 500k)"] + list(DETVAR_VARIATIONS.values())[1:]
SAMPLE_LATEX = {
    "data": "Data", "ext": "EXT", "nuwro": "NuWro Fake Data",
    "nu_overlay": r"$\nu$ Overlay", "nue_overlay": r"$\nu_e$CC Overlay",
    "nc_pi0_overlay": r"NC $\pi^0$ Overlay", "numucc_pi0_overlay": r"$\nu_\mu$CC $\pi^0$ Overlay",
    "dirt_overlay": "Dirt Overlay", "del1g_overlay": r"Del $1\gamma$ Overlay", "iso1g_overlay": r"Iso $1\gamma$ Overlay",
}
VARIATION_LATEX = {
    "CV (3b 1M)": r"CV (1M)$^*$", "CV (3b 500k)": r"CV (500k)$^\dagger$",
    "WireMod thetaXZ": r"WireMod $\theta_{XZ}$", "WireMod thetaYZ": r"WireMod $\theta_{YZ}$",
}

def fmt_pot(x):
    m, e = f"{x:.3e}".split("e")
    return f"${m}\\times10^{{{int(e)}}}$"

def fmt_int(x):
    return f"{int(x):,}".replace(",", "{,}")

NUM_EVENTS_HEADER = "\\makecell{\\textbf{Num} \\\\ \\textbf{Events}}"

def filepath(p):
    return "\\filepath{" + p + "}"

def table_body(df, group_col, cell_fn):
    # \midrule between groups, \hline within a group
    lines, prev = [], None
    for _, r in df.iterrows():
        if prev is not None:
            lines.append("\\midrule" if r[group_col] != prev else "\\hline")
        lines.append(" & ".join(cell_fn(r)) + " \\\\")
        prev = r[group_col]
    return "\n".join(lines)

def small_table(colspec, caption, label, header, body):
    return "\n".join([
        "\\begin{table}[H]", "\\centering", "\\tiny",
        "\\caption{" + caption + "}", "\\label{" + label + "}",
        "\\begin{adjustbox}{max width=\\textwidth}", "\\begin{tabular}{" + colspec + "}",
        "\\toprule", header + " \\\\", "\\midrule",
        body,
        "\\bottomrule", "\\end{tabular}", "\\end{adjustbox}", "\\end{table}",
    ])

def long_table(ncols, colspec, caption, label, header, body):
    return "\n".join([
        "{\\tiny", "\\begin{longtable}{" + colspec + "}",
        "\\caption{" + caption + "}", "\\label{" + label + "} \\\\",
        "\\toprule", header + " \\\\", "\\midrule", "\\endfirsthead",
        "\\toprule", header + " \\\\", "\\midrule", "\\endhead",
        "\\midrule", "\\multicolumn{" + str(ncols) + "}{r}{\\textit{Continued on next page}} \\\\", "\\endfoot",
        "\\bottomrule", "\\endlastfoot",
        body,
        "\\end{longtable}", "}",
    ])

def sort_df(df, order_cols):
    keys = []
    for col, order in order_cols:
        df = df.assign(**{"_" + col: df[col].map(order.index)})
        keys.append("_" + col)
    return df.sort_values(keys)

tables = []

# data and EXT
df = sort_df(pot_df[pot_df["sample"].isin(["data", "ext"])], [("sample", ["data", "ext"]), ("run", RUN_ORDER)])
tables.append(small_table(
    "lllllp{10cm}",
    "Data and EXT samples used in this analysis. Data POT and trigger counts (\\texttt{tor875\\_wcut} and "
    "\\texttt{E1DCNT\\_wcut}) and EXT trigger counts (\\texttt{EXT}) are from \\texttt{getDataInfo.py}. EXT POT "
    "values are POT-equivalents, equal to the EXT trigger count times the POT per trigger of the paired data period "
    "(run 1 for EXT run 1, run 3 for EXT runs 2 and 3, run 4a for EXT run 4a, and run 4b for EXT runs 4b, 4c, 4d, and 5).",
    "tab:data_ext_samples",
    "\\textbf{Type} & \\textbf{Run} & \\textbf{POT} & \\makecell{\\textbf{Num} \\\\ \\textbf{Triggers}} & " + NUM_EVENTS_HEADER + " & \\textbf{File Path}",
    table_body(df, "sample", lambda r: [SAMPLE_LATEX[r["sample"]], r["run"], fmt_pot(r["POT"]),
                                        fmt_int(r["num_triggers"]), fmt_int(r["n_events"]), filepath(r["path"])]),
))

# NuWro fake data
df = sort_df(pot_df[pot_df["sample"] == "nuwro"], [("run", RUN_ORDER)])
tables.append(small_table(
    "llllp{10cm}",
    "NuWro fake data samples used in this analysis. POT values are from \\texttt{pot\\_tor875good}.",
    "tab:fake_data_samples",
    "\\textbf{Type} & \\textbf{Run} & \\textbf{POT} & " + NUM_EVENTS_HEADER + " & \\textbf{File Path}",
    table_body(df, "sample", lambda r: [SAMPLE_LATEX[r["sample"]], r["run"], fmt_pot(r["POT"]), fmt_int(r["n_events"]), filepath(r["path"])]),
))

# MC overlays
mc_order = ["nu_overlay", "nue_overlay", "nc_pi0_overlay", "numucc_pi0_overlay", "dirt_overlay", "del1g_overlay", "iso1g_overlay"]
df = sort_df(pot_df[pot_df["sample"].isin(mc_order) & ~pot_df["detvar"]], [("sample", mc_order), ("run", RUN_ORDER)])
tables.append(long_table(
    5, "llllp{8.5cm}",
    "Monte Carlo overlay samples used in this analysis. POT values are from \\texttt{pot\\_tor875good}.",
    "tab:mc_samples",
    "\\textbf{Type} & \\textbf{Run} & \\textbf{POT} & " + NUM_EVENTS_HEADER + " & \\textbf{File Path}",
    table_body(df, "sample", lambda r: [SAMPLE_LATEX[r["sample"]], r["run"], fmt_pot(r["POT"]), fmt_int(r["n_events"]), filepath(r["path"])]),
))

# DetVar, grouped by sample and variation (all CV flavors in one group)
df = pot_df[pot_df["detvar"]].copy()
df["variation_group"] = df["variation"].str.replace(r"^CV.*", "CV", regex=True)
df["group"] = df["sample"] + "/" + df["variation_group"]
df = sort_df(df, [("sample", ["nu_overlay", "nue_overlay"]), ("variation_group", VARIATION_ORDER),
                  ("run", RUN_ORDER), ("variation", VARIATION_ORDER)])
tables.append(long_table(
    6, "lllllp{7.5cm}",
    "Detector variation samples used in this analysis. POT values are from \\texttt{pot\\_tor875good}. "
    "The run 1 and run 3a CVs are the two halves of the \\texttt{13a} CV sample, and pair with the run 1 and run 3a "
    "events of every variation. For run 3b events, $^*$the 1M-event CV sample pairs with all variations except SCE "
    "and Recombination, and $^\\dagger$the 500k-event CV sample pairs with only SCE and Recombination.",
    "tab:detvar_samples",
    "\\textbf{File Type} & \\textbf{Variation} & \\textbf{Run} & \\textbf{POT} & " + NUM_EVENTS_HEADER + " & \\textbf{File Path}",
    table_body(df, "group", lambda r: [SAMPLE_LATEX[r["sample"]], VARIATION_LATEX.get(r["variation"], r["variation"]),
                                       r["run"], fmt_pot(r["POT"]), fmt_int(r["n_events"]), filepath(r["path"])]),
))

n_in_tables = sum(t.count("\\filepath{") for t in tables)
assert n_in_tables == len(pot_df), (n_in_tables, len(pot_df))
generated = "\n\n".join(tables)
print(generated)

In [ ]:
# write the generated tables into the tex file, between the markers
BEGIN = "% BEGIN AUTO-GENERATED FILE TABLES (ipynb_notebooks/file_pot_tables.ipynb)"
END = "% END AUTO-GENERATED FILE TABLES"

tex = open(tex_file).read()
i, j = tex.index(BEGIN), tex.index(END)
tex = tex[:i] + BEGIN + "\n\n" + generated + "\n\n" + tex[j:]
open(tex_file, "w").write(tex)
print(f"wrote {len(pot_df)} files into {tex_file}")